# Participación regional de la demanda residencial

Procesa `Insumos/Salida Compilada - Residencial (Energía útil) - Sin subtotales-2.xlsx` (una hoja por región), limpia la ruta de tecnología separando solo por `\` y calcula tres participaciones porcentuales regionales:

1. Por **tecnología completa** (`Zona\Uso\fuel\eficiencia`) y `Año`.
2. Por **FUEL de OSeMOSYS** (`RESCLIM_URB`, `RESDHT_RUR`, ...) y `Año`: cada ruta se mapea a un código según su Uso (segundo segmento) o Uso + tercer segmento, con sufijo `_URB`/`_RUR` según la Zona.
3. Por **TECHNOLOGY de OSeMOSYS** (`DEMRESELCWHT_PAS_LOW_URB`, ...) y `Año`: cada TECHNOLOGY del SAND se mapea a su familia tecnológica LEAP **agregando las tres eficiencias** (`Eficiencia_existente`, `Mejor eficiencia_Colombia`, `Mejor eficiencia_internacional`), de modo que las variantes `LOW`/`MID`/`HIG` comparten la misma distribución regional.

Diferencias frente al notebook industrial:

- **NO se elimina el primer segmento** de la ruta: aquí no es un `Subsector {Código}` sino la **Zona** (`Rural`, `Urbano`, `ZNI`), que es parte de la identificación de la tecnología.
- Se **excluye la zona `ZNI`** (no se usa en el modelo) y las filas `Total`.
- La agrupación resumida es por **FUEL de OSeMOSYS** (Zona + Uso, y en algunos casos también el tercer segmento), no por el combustible LEAP.

Se omite el año 2021 y los totales nacionales nulos o cero producen participación 0.

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', 200)

In [ ]:
# --- Configuración ---
INPUT_PATH = "Insumos/Salida Compilada - Residencial (Energía útil) - Sin subtotales-2.xlsx"
OUTPUT_PATH = "Insumos/Participacion_Regional_Residencial.xlsx"

HEADER_ROW = 5          # fila 0-indexada donde está el encabezado real ("Branch", 2021, 2022, ...)
ANIO_EXCLUIDO = 2021
COL_RUTA = "Branch"
ZONAS_EXCLUIDAS = {"ZNI", "Total"}   # ZNI no se usa; 'Total' son filas de subtotal

# Sufijo del FUEL OSeMOSYS según la Zona (primer segmento de la ruta)
SUFIJO_ZONA = {"Urbano": "URB", "Rural": "RUR"}

# Mapeo a FUEL OSeMOSYS por (Uso, tercer segmento) — casos donde el Uso se reparte en varios fuels
MAPEO_USO_TERCERO = {
    ("Refrigeracion", "Aire Acondicionado"): "RESCLIM",
    ("Fuerza Motriz", "Ventiladores"): "RESCLIM",
    ("Refrigeracion", "Neveras"): "RESREF",
    ("Fuerza Motriz", "Lavadora"): "RESWSH",
}

# Mapeo a FUEL OSeMOSYS por Uso completo (segundo segmento), cuando no aplica el anterior
MAPEO_USO = {
    "Calor Directo": "RESDHT",
    "Iluminacion": "RESILU",
    "Otros": "RESOTH",
    "TV": "RESTV",
    "Calentamiento Agua": "RESWHT",
}

## 1. Lectura de todas las hojas (regiones)

In [ ]:
hojas = pd.read_excel(INPUT_PATH, sheet_name=None, skiprows=HEADER_ROW)
print("Regiones encontradas:", list(hojas.keys()))

primera = next(iter(hojas.values()))
print("Columnas de ejemplo:", primera.columns.tolist())

## 2. Limpieza de la ruta de tecnología y filtro de años

Toda la limpieza se hace separando por el carácter `\` (sin límites de longitud ni índices fijos):

1. **Reparación de truncamiento** (defensiva): el exporte de LEAP limita la ruta a 100 caracteres. En este archivo ninguna ruta llega al límite, pero se conserva la lógica por si un re-exporte futuro trae segmentos cortados: se comparan los segmentos finales contra los nombres canónicos observados y se restaura el texto completo.
2. **La ruta se conserva completa** (`Zona\Uso\fuel\eficiencia`): el primer segmento es la Zona (`Rural`/`Urbano`), no un subsector a descartar.
3. Se excluyen las filas cuya zona esté en `ZONAS_EXCLUIDAS` (`ZNI` y los subtotales `Total`), la columna `2021` y cualquier columna auxiliar como `Total`.
4. Se agrupan (sumando) las rutas que queden duplicadas dentro de cada hoja.

In [ ]:
def construir_mapa_reparacion(hojas: dict) -> dict:
    """Mapa {segmento_final_truncado: nombre_canonico_completo}.

    El origen limita la ruta a 100 caracteres y puede cortar el último
    segmento (ej. 'Mejor eficiencia_internaciona'). Un segmento es canónico
    si NO es prefijo estricto de otro segmento observado; cada segmento
    truncado se mapea al único canónico que lo contiene como prefijo.
    """
    finales = set()
    for df in hojas.values():
        finales |= {str(r).split("\\")[-1] for r in df[COL_RUTA].dropna()}

    canonicos = [s for s in finales if not any(o != s and o.startswith(s) for o in finales)]

    mapa = {}
    for seg in finales:
        candidatos = [c for c in canonicos if c.startswith(seg)]
        if len(candidatos) == 1:
            mapa[seg] = candidatos[0]
    return mapa


MAPA_REPARACION = construir_mapa_reparacion(hojas)
reparados = {k: v for k, v in MAPA_REPARACION.items() if k != v}
print(f"Segmentos truncados reparados: {len(reparados)}")
for trunc, completo in sorted(reparados.items()):
    print(f"  {trunc!r} -> {completo!r}")


def limpiar_ruta(ruta: str) -> str:
    """Repara el segmento final truncado y devuelve la ruta COMPLETA
    (Zona\\Uso\\fuel\\eficiencia). A diferencia del notebook industrial,
    NO se elimina el primer segmento: es la Zona (Rural/Urbano)."""
    partes = str(ruta).split("\\")
    partes[-1] = MAPA_REPARACION.get(partes[-1], partes[-1])
    return "\\".join(partes)


def procesar_hoja(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Descartar filas sin ruta, filas de totales y zonas excluidas (ZNI)
    df = df[df[COL_RUTA].notna()]
    partes = df[COL_RUTA].astype(str).str.split("\\")
    df = df[(partes.str.len() >= 2) & ~partes.str[0].isin(ZONAS_EXCLUIDAS)]

    # Columnas de año: numéricas y dentro del rango válido (excluye 2021 y cualquier 'Total')
    year_cols = [
        c for c in df.columns
        if isinstance(c, (int, float)) and not pd.isna(c) and int(c) != ANIO_EXCLUIDO
    ]

    df["Tecnologia"] = df[COL_RUTA].apply(limpiar_ruta)

    df = df[["Tecnologia"] + year_cols]
    df.columns = ["Tecnologia"] + [int(c) for c in year_cols]

    # Agrupar rutas duplicadas tras la limpieza, sumando valores por año
    df = df.groupby("Tecnologia", as_index=False).sum(numeric_only=True)
    return df


hojas_limpias = {region: procesar_hoja(df) for region, df in hojas.items()}
hojas_limpias["Antioquia"].head()

## 3. Formato largo y consolidación en un DataFrame maestro

In [ ]:
def a_formato_largo(region: str, df: pd.DataFrame) -> pd.DataFrame:
    year_cols = [c for c in df.columns if c != "Tecnologia"]
    largo = df.melt(id_vars="Tecnologia", value_vars=year_cols, var_name="Anio", value_name="Valor")
    largo.insert(0, "Region", region)
    return largo


df_maestro = pd.concat(
    [a_formato_largo(region, df) for region, df in hojas_limpias.items()],
    ignore_index=True,
)
df_maestro["Anio"] = df_maestro["Anio"].astype(int)
df_maestro["Valor"] = pd.to_numeric(df_maestro["Valor"], errors="coerce").fillna(0)

print(df_maestro.shape)
df_maestro.head()

## 4. Total nacional por Tecnología y Año

In [ ]:
df_total_nacional = (
    df_maestro.groupby(["Tecnologia", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)
df_total_nacional.head()

## 5. Cálculo de participación porcentual por región

In [ ]:
df_participacion = pd.merge(df_maestro, df_total_nacional, on=["Tecnologia", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion["Total_Nacional"] = df_participacion["Total_Nacional"].fillna(0)
df_participacion["Participacion"] = 0.0
mask_valido = df_participacion["Total_Nacional"] != 0
df_participacion.loc[mask_valido, "Participacion"] = (
    df_participacion.loc[mask_valido, "Valor"] / df_participacion.loc[mask_valido, "Total_Nacional"]
)

df_participacion = df_participacion.sort_values(["Tecnologia", "Anio", "Region"]).reset_index(drop=True)
df_participacion.head(20)

## 6. Salida: regiones como columnas, indexado por Tecnología y Año

In [ ]:
df_salida = df_participacion.pivot_table(
    index=["Tecnologia", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida.columns.name = None
df_salida = df_salida.reset_index()
df_salida.head(20)

In [ ]:
# Verificación: la suma de participaciones por Tecnologia-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols = [c for c in df_salida.columns if c not in ("Tecnologia", "Anio")]
suma_check = df_salida[region_cols].sum(axis=1)
filas_invalidas = ((suma_check - 1).abs() > 1e-6) & (suma_check.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas.sum()} de {len(df_salida)}")

## 7. Participación regional por FUEL de OSeMOSYS

Cada ruta `Zona\Uso\tercero\...` se mapea a un **FUEL de OSeMOSYS** `RES{USO}_{ZONA}`:

| FUEL | Corresponde a |
|---|---|
| `RESCLIM` | `Refrigeracion\Aire Acondicionado` + `Fuerza Motriz\Ventiladores` |
| `RESDHT`  | `Calor Directo` (todos los combustibles) |
| `RESILU`  | `Iluminacion` |
| `RESOTH`  | `Otros` |
| `RESREF`  | `Refrigeracion\Neveras` |
| `RESTV`   | `TV` |
| `RESWHT`  | `Calentamiento Agua` |
| `RESWSH`  | `Fuerza Motriz\Lavadora` |

El sufijo es `_URB` para `Urbano` y `_RUR` para `Rural`. El mapeo intenta primero por `(Uso, tercer segmento)` (`MAPEO_USO_TERCERO`, los casos donde un mismo Uso se reparte en varios fuels) y si no aplica, por el `Uso` completo (`MAPEO_USO`). Cualquier ruta que no encaje en ningún mapeo detiene el notebook con error.

Se agrupan los valores por `Region`, `Fuel` y `Anio`, se calcula el total nacional por FUEL y la participación de cada región.

In [ ]:
def mapear_fuel_osemosys(tecnologia: str):
    """Devuelve el FUEL OSeMOSYS (ej. 'RESCLIM_URB') para una ruta
    'Zona\\Uso\\tercero\\...', o None si no hay mapeo."""
    partes = tecnologia.split("\\")
    zona = SUFIJO_ZONA.get(partes[0])
    if zona is None or len(partes) < 2:
        return None
    uso = partes[1]
    tercero = partes[2] if len(partes) >= 3 else None
    codigo = MAPEO_USO_TERCERO.get((uso, tercero)) or MAPEO_USO.get(uso)
    return f"{codigo}_{zona}" if codigo else None


df_fuel = df_maestro.copy()
df_fuel["Fuel"] = df_fuel["Tecnologia"].map(mapear_fuel_osemosys)

sin_mapeo = sorted(df_fuel.loc[df_fuel["Fuel"].isna(), "Tecnologia"].unique())
if sin_mapeo:
    for t in sin_mapeo:
        print(f"  SIN MAPEO: {t}")
    raise ValueError(f"{len(sin_mapeo)} tecnologías sin FUEL OSeMOSYS asignado")

print("FUELs OSeMOSYS encontrados:", sorted(df_fuel["Fuel"].unique()))

# Detalle del mapeo aplicado (para revisión): rutas Zona\Uso[\tercero] -> FUEL
detalle_mapeo = (
    df_fuel.assign(Ruta=df_fuel["Tecnologia"].str.split("\\").str[:3].str.join("\\"))
    [["Ruta", "Fuel"]].drop_duplicates().sort_values(["Fuel", "Ruta"]).reset_index(drop=True)
)
detalle_mapeo

In [ ]:
# Agrupar por Region, Fuel OSeMOSYS y Anio
df_fuel = df_fuel.groupby(["Region", "Fuel", "Anio"], as_index=False)["Valor"].sum()

# Total nacional a nivel de FUEL y participación regional
df_total_fuel = (
    df_fuel.groupby(["Fuel", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)

df_participacion_fuel = pd.merge(df_fuel, df_total_fuel, on=["Fuel", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion_fuel["Total_Nacional"] = df_participacion_fuel["Total_Nacional"].fillna(0)
df_participacion_fuel["Participacion"] = 0.0
mask_valido = df_participacion_fuel["Total_Nacional"] != 0
df_participacion_fuel.loc[mask_valido, "Participacion"] = (
    df_participacion_fuel.loc[mask_valido, "Valor"]
    / df_participacion_fuel.loc[mask_valido, "Total_Nacional"]
)

df_participacion_fuel = (
    df_participacion_fuel.sort_values(["Fuel", "Anio", "Region"]).reset_index(drop=True)
)
df_participacion_fuel.head(20)

In [ ]:
# Salida por FUEL: regiones como columnas, indexado por Fuel y Año
df_salida_fuel = df_participacion_fuel.pivot_table(
    index=["Fuel", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida_fuel.columns.name = None
df_salida_fuel = df_salida_fuel.reset_index()

# Verificación: la suma de participaciones por Fuel-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols_fuel = [c for c in df_salida_fuel.columns if c not in ("Fuel", "Anio")]
suma_check_fuel = df_salida_fuel[region_cols_fuel].sum(axis=1)
filas_invalidas_fuel = ((suma_check_fuel - 1).abs() > 1e-6) & (suma_check_fuel.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas_fuel.sum()} de {len(df_salida_fuel)}")
df_salida_fuel.head(20)

## 8. Participación regional por TECHNOLOGY de OSeMOSYS

Cada TECHNOLOGY del SAND se mapea a las rutas LEAP de su **familia tecnológica**. Convenciones:

- **Eficiencia**: la participación se calcula **agregando las tres eficiencias** (`Eficiencia_existente`, `Mejor eficiencia_Colombia`, `Mejor eficiencia_internacional`): se suma la demanda de la familia completa antes de sacar porcentajes, así que las variantes `LOW`/`MID`/`HIG` de una misma tecnología comparten la misma distribución regional.
- **Zona**: `_RUR` = `Rural`, `_URB` = `Urbano`. Las tecnologías `DEMRESZNI*` se marcan como excluidas (ZNI no se usa).
- **Aire acondicionado**: `SPL` = `Mini Split`, `PAR` = `Pared o ventana`, `POR` = `Central`.
- **Calentamiento agua eléctrico**: `PAS` = `Electricidad de paso`, `DUC` = `Electricidad ducha`, `TAN` = `Electricidad_tanque`.
- **Gas natural WHT**: LEAP no separa `FOR`/`NAT`; ambas tecnologías toman la distribución de la única rama `Gas natural tanques y de paso`.
- **Otros**: la rama `Otros\All others` no tiene niveles de eficiencia; los tres niveles (`LOW`/`MID`/`HIG`) comparten su distribución.
- **TV**: `CRT` agrega las tres eficiencias de `Convencional CRT`; las variantes `LOW`/`MID`/`HIG` comparten la distribución agregada de `LCD Plasma LED` (rural) / `LED LCD` (urbano).
- **Iluminación**: rural agregada sobre las eficiencias de `Iluminacion UE`; urbana por bombillo: `INC` agrega `Incandescente 60W` + `incandescente 50W`, `LFC` = `LFC`. Los bombillos `LED` y `Fluorecente` no tienen TECHNOLOGY y las `DEMRESELCILU_{LOW,MID,HIG}_URB` e `_{INC,LFC}_RUR` no tienen rama LEAP: quedan reportadas como `SIN_MAPEO` / `RAMA_SIN_TECHNOLOGY`.
- Las ramas `Calor Directo\Carbon` y `Calor Directo\Kerosene` no tienen TECHNOLOGY en el SAND; `DEMRESBGSCKN_MID_RUR` (biogás) y `DEMRES_MEDPVA_URB` no tienen rama LEAP. Todo queda reportado en la hoja `Mapeo_Technology`.

Cuando una TECHNOLOGY agrupa varias rutas, primero se suman los valores y luego se calcula la participación.

In [ ]:
TECNOLOGIAS_OSEMOSYS = """
DEMRES_MEDPVA_URB
DEMRESBGSCKN_MID_RUR
DEMRESELCAIR_PAR_HIG_RUR DEMRESELCAIR_PAR_LOW_RUR DEMRESELCAIR_PAR_LOW_URB DEMRESELCAIR_PAR_MID_RUR DEMRESELCAIR_PAR_MID_URB
DEMRESELCAIR_POR_HIG_RUR DEMRESELCAIR_POR_HIG_URB DEMRESELCAIR_POR_LOW_RUR DEMRESELCAIR_POR_LOW_URB DEMRESELCAIR_POR_MID_RUR DEMRESELCAIR_POR_MID_URB
DEMRESELCAIR_SPL_HIG_RUR DEMRESELCAIR_SPL_HIG_URB DEMRESELCAIR_SPL_LOW_RUR DEMRESELCAIR_SPL_LOW_URB DEMRESELCAIR_SPL_MID_RUR DEMRESELCAIR_SPL_MID_URB
DEMRESELCCKN_HIG_RUR DEMRESELCCKN_HIG_URB DEMRESELCCKN_LOW_RUR DEMRESELCCKN_LOW_URB DEMRESELCCKN_MID_RUR DEMRESELCCKN_MID_URB
DEMRESELCFAN_HIG_RUR DEMRESELCFAN_HIG_URB DEMRESELCFAN_LOW_RUR DEMRESELCFAN_LOW_URB DEMRESELCFAN_MID_RUR DEMRESELCFAN_MID_URB
DEMRESELCILU_HIG_RUR DEMRESELCILU_HIG_URB DEMRESELCILU_INC_RUR DEMRESELCILU_INC_URB DEMRESELCILU_LFC_RUR DEMRESELCILU_LFC_URB
DEMRESELCILU_LOW_RUR DEMRESELCILU_LOW_URB DEMRESELCILU_MID_RUR DEMRESELCILU_MID_URB
DEMRESELCOTH_HIG_RUR DEMRESELCOTH_HIG_URB DEMRESELCOTH_LOW_RUR DEMRESELCOTH_LOW_URB DEMRESELCOTH_MID_RUR DEMRESELCOTH_MID_URB
DEMRESELCREF_HIG_RUR DEMRESELCREF_HIG_URB DEMRESELCREF_LOW_RUR DEMRESELCREF_LOW_URB DEMRESELCREF_MID_RUR DEMRESELCREF_MID_URB
DEMRESELCTV_CRT_RUR DEMRESELCTV_CRT_URB DEMRESELCTV_HIG_RUR DEMRESELCTV_HIG_URB DEMRESELCTV_LOW_RUR DEMRESELCTV_LOW_URB DEMRESELCTV_MID_RUR DEMRESELCTV_MID_URB
DEMRESELCWHT_DUC_HIG_RUR DEMRESELCWHT_DUC_HIG_URB DEMRESELCWHT_DUC_LOW_RUR DEMRESELCWHT_DUC_LOW_URB DEMRESELCWHT_DUC_MID_RUR DEMRESELCWHT_DUC_MID_URB
DEMRESELCWHT_PAS_HIG_RUR DEMRESELCWHT_PAS_HIG_URB DEMRESELCWHT_PAS_LOW_RUR DEMRESELCWHT_PAS_LOW_URB DEMRESELCWHT_PAS_MID_RUR DEMRESELCWHT_PAS_MID_URB
DEMRESELCWHT_TAN_HIG_RUR DEMRESELCWHT_TAN_HIG_URB DEMRESELCWHT_TAN_LOW_RUR DEMRESELCWHT_TAN_LOW_URB DEMRESELCWHT_TAN_MID_RUR DEMRESELCWHT_TAN_MID_URB
DEMRESELCWSH_HIG_RUR DEMRESELCWSH_HIG_URB DEMRESELCWSH_LOW_RUR DEMRESELCWSH_LOW_URB DEMRESELCWSH_MID_RUR DEMRESELCWSH_MID_URB
DEMRESLPGCKN_HIG_RUR DEMRESLPGCKN_HIG_URB DEMRESLPGCKN_LOW_RUR DEMRESLPGCKN_LOW_URB DEMRESLPGCKN_MID_RUR DEMRESLPGCKN_MID_URB
DEMRESNGSCKN_HIG_RUR DEMRESNGSCKN_HIG_URB DEMRESNGSCKN_LOW_RUR DEMRESNGSCKN_LOW_URB DEMRESNGSCKN_MID_RUR DEMRESNGSCKN_MID_URB
DEMRESNGSWHT_FOR_HIG_RUR DEMRESNGSWHT_FOR_HIG_URB DEMRESNGSWHT_FOR_LOW_RUR DEMRESNGSWHT_FOR_LOW_URB DEMRESNGSWHT_FOR_MID_RUR DEMRESNGSWHT_FOR_MID_URB
DEMRESNGSWHT_NAT_HIG_RUR DEMRESNGSWHT_NAT_HIG_URB DEMRESNGSWHT_NAT_LOW_RUR DEMRESNGSWHT_NAT_LOW_URB
DEMRESWOOCKN_HIG_RUR DEMRESWOOCKN_HIG_URB DEMRESWOOCKN_LOW_RUR DEMRESWOOCKN_LOW_URB DEMRESWOOCKN_MID_RUR DEMRESWOOCKN_MID_URB
DEMRESZNIBGSCKN_MID DEMRESZNIELC_LOW DEMRESZNIELCCKN_LOW DEMRESZNILPGCKN_MID DEMRESZNIWOOCKN_LOW
""".split()

EFICIENCIA = {
    "LOW": "Eficiencia_existente",
    "MID": "Mejor eficiencia_Colombia",
    "HIG": "Mejor eficiencia_internacional",
}
ZONA_TECH = {"RUR": "Rural", "URB": "Urbano"}

# Familias con eficiencia estándar: prefijo TECHNOLOGY -> ruta base LEAP (sin zona ni eficiencia)
FAMILIAS = {
    "DEMRESELCWHT_PAS": "Calentamiento Agua\\Electricidad\\Electricidad de paso",
    "DEMRESELCWHT_DUC": "Calentamiento Agua\\Electricidad\\Electricidad ducha",
    "DEMRESELCWHT_TAN": "Calentamiento Agua\\Electricidad\\Electricidad_tanque",
    "DEMRESNGSWHT_FOR": "Calentamiento Agua\\Gas natural tanques y de paso",  # LEAP no separa FOR/NAT:
    "DEMRESNGSWHT_NAT": "Calentamiento Agua\\Gas natural tanques y de paso",  # ambas usan la misma rama
    "DEMRESELCCKN": "Calor Directo\\Electricidad_SIN",
    "DEMRESLPGCKN": "Calor Directo\\GLP",
    "DEMRESNGSCKN": "Calor Directo\\Gas Natural",
    "DEMRESWOOCKN": "Calor Directo\\Lena",
    "DEMRESELCWSH": "Fuerza Motriz\\Lavadora",
    "DEMRESELCFAN": "Fuerza Motriz\\Ventiladores",
    "DEMRESELCREF": "Refrigeracion\\Neveras\\Neveras UE",
    "DEMRESELCAIR_SPL": "Refrigeracion\\Aire Acondicionado\\Mini Split",
    "DEMRESELCAIR_PAR": "Refrigeracion\\Aire Acondicionado\\Pared o ventana",
    "DEMRESELCAIR_POR": "Refrigeracion\\Aire Acondicionado\\Central",
}

# Generar candidatos {TECHNOLOGY: [rutas LEAP]}; luego se filtran a las tecnologías del SAND.
# La participación es independiente de la eficiencia: cada variante LOW/MID/HIG recibe las
# TRES ramas de eficiencia de su familia (se suma toda la demanda antes de sacar porcentajes).
candidatos = {}
for prefijo, base in FAMILIAS.items():
    for zc, zn in ZONA_TECH.items():
        rutas_familia = [f"{zn}\\{base}\\{seg}" for seg in EFICIENCIA.values()]
        for eff in EFICIENCIA:
            candidatos[f"{prefijo}_{eff}_{zc}"] = rutas_familia

for zc, zn in ZONA_TECH.items():
    # 'Otros' no tiene segmento de eficiencia: los tres niveles comparten la misma rama
    for eff in EFICIENCIA:
        candidatos[f"DEMRESELCOTH_{eff}_{zc}"] = [f"{zn}\\Otros\\All others"]
    # TV CRT: una sola TECHNOLOGY agrega las tres eficiencias LEAP
    candidatos[f"DEMRESELCTV_CRT_{zc}"] = [f"{zn}\\TV\\Convencional CRT\\{seg}" for seg in EFICIENCIA.values()]

# Iluminación rural y TV LCD/LED: familia agregada, misma distribución para LOW/MID/HIG
rutas_ilu_rur = [f"Rural\\Iluminacion\\Iluminacion UE\\{seg}" for seg in EFICIENCIA.values()]
rutas_tv_rur = [f"Rural\\TV\\LCD Plasma LED\\{seg}" for seg in EFICIENCIA.values()]
rutas_tv_urb = [f"Urbano\\TV\\LED LCD\\{seg}" for seg in EFICIENCIA.values()]
for eff in EFICIENCIA:
    candidatos[f"DEMRESELCILU_{eff}_RUR"] = rutas_ilu_rur
    candidatos[f"DEMRESELCTV_{eff}_RUR"] = rutas_tv_rur
    candidatos[f"DEMRESELCTV_{eff}_URB"] = rutas_tv_urb

# Iluminación urbana por tipo de bombillo: INC agrega ambas incandescentes
candidatos["DEMRESELCILU_INC_URB"] = [
    "Urbano\\Iluminacion\\Iluminacion Bombillos UE\\Incandescente 60W",
    "Urbano\\Iluminacion\\Iluminacion Bombillos UE\\incandescente 50W",
]
candidatos["DEMRESELCILU_LFC_URB"] = ["Urbano\\Iluminacion\\Iluminacion Bombillos UE\\LFC"]

MAPEO_TECHNOLOGY = {t: candidatos[t] for t in TECNOLOGIAS_OSEMOSYS if t in candidatos}
tec_zni = sorted(t for t in TECNOLOGIAS_OSEMOSYS if t.startswith("DEMRESZNI"))
tec_sin_mapeo = sorted(t for t in TECNOLOGIAS_OSEMOSYS if t not in candidatos and not t.startswith("DEMRESZNI"))

print(f"TECHNOLOGYs del SAND: {len(TECNOLOGIAS_OSEMOSYS)}")
print(f"  Mapeadas a rama LEAP: {len(MAPEO_TECHNOLOGY)}")
print(f"  ZNI (excluidas): {len(tec_zni)} -> {tec_zni}")
print(f"  Sin rama LEAP (SIN_MAPEO): {len(tec_sin_mapeo)}")
for t in tec_sin_mapeo:
    print(f"    {t}")

# Validación cruzada contra el insumo
rutas_insumo = set(df_maestro["Tecnologia"].unique())
rutas_mapeadas = {r for rutas in MAPEO_TECHNOLOGY.values() for r in rutas}
rutas_inexistentes = sorted(rutas_mapeadas - rutas_insumo)
assert not rutas_inexistentes, f"Rutas mapeadas que NO existen en el insumo: {rutas_inexistentes}"

rutas_sin_technology = sorted(rutas_insumo - rutas_mapeadas)
print(f"Ramas LEAP sin TECHNOLOGY asociada: {len(rutas_sin_technology)}")
for r in rutas_sin_technology:
    print(f"    {r}")

In [ ]:
# Valores por TECHNOLOGY: se suman las rutas LEAP asociadas antes de calcular la participación
df_map_tech = pd.DataFrame(
    [(tech, ruta) for tech, rutas in MAPEO_TECHNOLOGY.items() for ruta in rutas],
    columns=["Technology", "Tecnologia"],
)

df_tech = df_map_tech.merge(df_maestro, on="Tecnologia", how="left")
df_tech = df_tech.groupby(["Region", "Technology", "Anio"], as_index=False)["Valor"].sum()

# Total nacional por TECHNOLOGY y participación regional
df_total_tech = (
    df_tech.groupby(["Technology", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)

df_participacion_tech = pd.merge(df_tech, df_total_tech, on=["Technology", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion_tech["Total_Nacional"] = df_participacion_tech["Total_Nacional"].fillna(0)
df_participacion_tech["Participacion"] = 0.0
mask_valido = df_participacion_tech["Total_Nacional"] != 0
df_participacion_tech.loc[mask_valido, "Participacion"] = (
    df_participacion_tech.loc[mask_valido, "Valor"]
    / df_participacion_tech.loc[mask_valido, "Total_Nacional"]
)

df_participacion_tech = (
    df_participacion_tech.sort_values(["Technology", "Anio", "Region"]).reset_index(drop=True)
)
df_participacion_tech.head(20)

In [ ]:
# Salida por TECHNOLOGY: regiones como columnas, indexado por Technology y Año
df_salida_tech = df_participacion_tech.pivot_table(
    index=["Technology", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida_tech.columns.name = None
df_salida_tech = df_salida_tech.reset_index()

# Verificación: la suma de participaciones por Technology-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols_tech = [c for c in df_salida_tech.columns if c not in ("Technology", "Anio")]
suma_check_tech = df_salida_tech[region_cols_tech].sum(axis=1)
filas_invalidas_tech = ((suma_check_tech - 1).abs() > 1e-6) & (suma_check_tech.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas_tech.sum()} de {len(df_salida_tech)}")
df_salida_tech.head(20)

In [ ]:
# Hoja de trazabilidad del mapeo TECHNOLOGY <-> rama LEAP (incluye lo no mapeado en ambos sentidos)
detalle_technology = pd.concat(
    [
        df_map_tech.rename(columns={"Tecnologia": "Ruta_LEAP"}).assign(Estado="MAPEADA"),
        pd.DataFrame({"Technology": tec_sin_mapeo, "Ruta_LEAP": "", "Estado": "SIN_MAPEO"}),
        pd.DataFrame({"Technology": tec_zni, "Ruta_LEAP": "", "Estado": "ZNI_EXCLUIDA"}),
        pd.DataFrame({"Technology": "", "Ruta_LEAP": rutas_sin_technology, "Estado": "RAMA_SIN_TECHNOLOGY"}),
    ],
    ignore_index=True,
).sort_values(["Estado", "Technology", "Ruta_LEAP"], key=lambda s: s.astype(str)).reset_index(drop=True)
detalle_technology

## 9. Exportar resultado

Un solo archivo Excel con ocho hojas: participación por tecnología completa (`Zona\Uso\fuel\eficiencia`), por FUEL de OSeMOSYS y por TECHNOLOGY de OSeMOSYS, cada una en formato pivote (regiones como columnas) y plano, más las hojas de trazabilidad de ambos mapeos.

In [ ]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    # Cálculo 1: participación por tecnología completa (Zona\Uso\fuel\eficiencia)
    df_salida.to_excel(writer, sheet_name="Participacion_Regional", index=False)
    df_participacion.to_excel(writer, sheet_name="Participacion_Plano", index=False)
    # Cálculo 2: participación por FUEL de OSeMOSYS (RESCLIM_URB, RESDHT_RUR, ...)
    df_salida_fuel.to_excel(writer, sheet_name="Participacion_Fuel", index=False)
    df_participacion_fuel.to_excel(writer, sheet_name="Participacion_Fuel_Plano", index=False)
    # Cálculo 3: participación por TECHNOLOGY de OSeMOSYS (DEMRES...)
    df_salida_tech.to_excel(writer, sheet_name="Participacion_Technology", index=False)
    df_participacion_tech.to_excel(writer, sheet_name="Participacion_Tech_Plano", index=False)
    # Trazabilidad de los mapeos
    detalle_mapeo.to_excel(writer, sheet_name="Mapeo_Fuel", index=False)
    detalle_technology.to_excel(writer, sheet_name="Mapeo_Technology", index=False)

print(f"Archivo exportado en: {OUTPUT_PATH}")

# Verificación final: ninguna zona excluida debe quedar en la salida y los 16 fuels esperados están presentes
zonas_salida = {t.split("\\")[0] for t in df_salida["Tecnologia"].unique()}
print(f"Zonas en la salida: {sorted(zonas_salida)}")
assert not (zonas_salida & ZONAS_EXCLUIDAS), "Hay zonas excluidas en la salida"

fuels_esperados = {f"{c}_{z}" for c in set(MAPEO_USO.values()) | set(MAPEO_USO_TERCERO.values())
                   for z in SUFIJO_ZONA.values()}
fuels_salida = set(df_salida_fuel["Fuel"].unique())
faltantes = fuels_esperados - fuels_salida
print(f"FUELs en la salida: {len(fuels_salida)} de {len(fuels_esperados)} esperados")
if faltantes:
    print(f"  Faltantes: {sorted(faltantes)}")

techs_salida = set(df_salida_tech["Technology"].unique())
print(f"TECHNOLOGYs con participación: {len(techs_salida)} de {len(MAPEO_TECHNOLOGY)} mapeadas")
assert techs_salida == set(MAPEO_TECHNOLOGY), "Hay TECHNOLOGYs mapeadas sin participación calculada"